# 2D time reversal and adjoint

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elma16/beamax/blob/v0.4.0/examples/reconstruction/2d_time_reversal_and_adjoint.ipynb)


In [ ]:
# Install beamax for Google Colab. Safe to skip when running locally.
%pip install --quiet "beamax[kwave,viz-mpl] @ git+https://github.com/elma16/beamax.git@v0.4.0"

## Setup

In [ ]:
import jax

from beamax._example_pat import (
    box_scanner_acquisition,
    box_scanner_msgb_inputs,
    homogeneous_msgb,
    timed,
    show_and_save,
)
from beamax.plotter import plot_packet_box_scanner

jax.config.update("jax_enable_x64", jax.default_backend() != "tpu")

## Acquire the left-face recording with k-Wave

In [ ]:
# The two-packet source occupies a homogeneous, open 128 × 128 domain.
p0, domain, face, ts, kwave, record = box_scanner_acquisition()
cropped, ts_data, wpt = box_scanner_msgb_inputs(record, ts)
msgb = homogeneous_msgb(1849)

## Time-reverse the recording with k-Wave and MSGB

In [ ]:
tr_kw = timed(
    "k-Wave TR",
    kwave.time_reversal,
    record,
    domain,
    detectors=face,
    ts=ts,
)
tr_msgb = timed(
    "MSGB TR",
    msgb.time_reversal,
    cropped,
    domain,
    detectors=face,
    ts=ts_data,
    data_wpt=wpt,
)
tr = (tr_kw, tr_msgb)

## Apply the k-Wave and MSGB surface adjoints

In [ ]:
adj_kw = timed(
    "k-Wave adjoint",
    kwave.adjoint,
    record,
    domain,
    detectors=face,
    ts=ts,
    pairing="surface_area",
)
adj_msgb = timed(
    "MSGB adjoint",
    msgb.adjoint,
    cropped,
    domain,
    detectors=face,
    ts=ts_data,
    data_wpt=wpt,
    pairing="surface_area",
)
adj = (adj_kw, adj_msgb)

## Plot the one-face fields at the thesis scale and marked column

In [ ]:
fig = plot_packet_box_scanner(
    p0,
    face.to_mask(domain).binary_mask,
    domain.grid_size,
    time_reversal=tr,
    adjoint=adj,
)
show_and_save(fig, "plots/reconstruction/2d_time_reversal_and_adjoint.png")